In [1]:
import numpy as np
import pandas as pd
import random
import torch
import os
import holidays
from typing import Iterable, Optional
from sklearn.model_selection import BaseCrossValidator
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import make_scorer
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna
from sklearn.base import clone
from sklearn.model_selection import cross_val_score
import json
from pytorch_forecasting.data import TimeSeriesDataSet, GroupNormalizer
from pytorch_forecasting.models import TemporalFusionTransformer
from pytorch_forecasting.metrics import SMAPE, QuantileLoss
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch import Trainer, seed_everything
from sklearn.linear_model import RidgeCV

os.environ['PYTHONHASHSEED'] = str(72)
torch.set_float32_matmul_precision('high')

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchmetrics\utilities\imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


# Configuration

In [2]:
# ==================================================
# Configuration
# ==================================================

CFG = {
    'SEED': 72,
    'LOOKBACK': 28,
    'PREDICT': 7,
    'N_CLUSTERS': 2,
    'K_VALIDATIONS': 5,
    'BATCH': 1024,
    'EPOCH': 100,
    'DEVICE': 'gpu' if torch.cuda.is_available() else 'cpu'
}

# Functions & Classes

In [3]:
# ==================================================
# Seed Locker
# ==================================================

def SeedLocker(seed = CFG['SEED']):
    seed_everything(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [4]:
# ==================================================
# Feature Extractor for Tree-Based Models
# ==================================================

def TreeFeatures(df: pd.DataFrame):
    # Column Organization
    df = df.rename(columns = {'영업일자': 'date', '영업장명_메뉴명': 'store_menu', '매출수량': 'qty'})
    df[['store', 'menu']] = df['store_menu'].str.split('_', n = 1, expand = True, regex = False)
    df['date'] = pd.to_datetime(df['date'], format = '%Y-%m-%d')

    # Quarter: 1, 2, 3, 4
    df['quarter'] = df['date'].dt.quarter

    # Month: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12
    df['month'] = df['date'].dt.month

    # Day of the Week: 1, 2, 3, 4, 5, 6, 7
    df['day_of_week'] = df['date'].dt.day_of_week + 1

    # Week of the Year: 1, ..., 53
    iso = df['date'].dt.isocalendar()
    df['week_of_year'] = iso.week.astype(int)

    # Season: 1, 2, 3, 4
    df['season'] = np.select([
        df['month'].isin([3, 4, 5]),
        df['month'].isin([6, 7, 8]),
        df['month'].isin([9, 10, 11]),
        df['month'].isin([12, 1, 2])
    ], [1, 2, 3, 4])

    # Peak: 0-1
    df['is_peak'] = df['month'].isin([1, 2]).astype(int)

    # Weekend: 0-1
    df['is_weekend'] = df['day_of_week'].isin([6, 7]).astype(int)

    # Holiday: 0-1
    years = df['date'].dt.year.unique()
    df['is_holiday'] = (df['date'].dt.date.isin(holidays.KR(years = years))).astype(int)
    
    # Potential Group Menu: 0-1
    df['potential_group_menu'] = df['menu'].str.contains(
        '단체|Conference|room|OPUS|이용료|Cookie Platter', regex = True
    ).astype(int)

    # Holidays Left in the Year
    df_list = []
    for year in years:
        df_tmp = pd.DataFrame({'date': pd.date_range(start = f'{year}-01-01', end = f'{year}-12-31')})
        df_tmp['is_holiday'] = (df_tmp['date'].dt.date.isin(holidays.KR(years = [year])))
        df_tmp['cum_holiday'] = df_tmp['is_holiday'].cumsum()
        df_tmp['holidays_left_in_year'] = (
            df_tmp['cum_holiday'].max()
            - df_tmp['cum_holiday']
            + df_tmp['is_holiday']
        )
        df_list.append(df_tmp)
    df_tmp = pd.concat(df_list, axis = 0)
    df = df.merge(df_tmp[['date', 'holidays_left_in_year']], 'left', 'date')

    # Isolated Holiday: 0-1
    df_tmp = pd.DataFrame({'date': pd.date_range(
            start = df['date'].min() - pd.Timedelta(days = 30),
            end = df['date'].max() + pd.Timedelta(days = 30)
    )})
    df_tmp['is_holiday'] = df_tmp['date'].dt.date.isin(holidays.KR(years = df_tmp['date'].dt.year.unique()))
    df_tmp['prev_holiday'] = df_tmp['is_holiday'].shift(1, fill_value = 0)
    df_tmp['next_holiday'] = df_tmp['is_holiday'].shift(-1, fill_value = 0)
    df_tmp['isolated_holiday'] = (
        (df_tmp['date'].dt.day_of_week.isin([1, 2, 3])) &
        (df_tmp['is_holiday'] == 1) &
        (df_tmp['prev_holiday'] == 0) &
        (df_tmp['next_holiday'] == 0)).astype(int)
    df = df.merge(df_tmp[['date', 'isolated_holiday']], 'left', 'date')

    # Holidays Left in Streak
    series_tmp = df_tmp.groupby('date', as_index = True)['is_holiday'].max().sort_index()
    series_tmp_rev = series_tmp.iloc[::-1]
    series_tmp = (
        series_tmp_rev.cumsum() - series_tmp_rev.cumsum().where(~series_tmp_rev).ffill().fillna(0)
    ).iloc[::-1]
    df_tmp = series_tmp.rename('holidays_left_in_streak').reset_index().rename(columns = {'index': 'date'})
    df = df.merge(df_tmp, 'left', 'date')
    df['holidays_left_in_streak'] = df['holidays_left_in_streak'].astype(int)

    # Keep Default Sorting
    df = df.sort_values(['store_menu', 'date']).reset_index(drop = True)

    # Lag-7, ..., 28
    for t in range(7, 29):
        df[f'lag_{t}d'] = df.groupby('store_menu')['qty'].shift(t)

    # Mean/SD/Median of Past Weeks
    week_blocks = {1: list(range(7, 14)), 2: list(range(14, 21)), 3: list(range(21, 28))}
    for w, lags in week_blocks.items():
        cols = [f'lag_{i}d' for i in lags]
        df[f'mean_{w}w_ago'] = df[cols].mean(axis = 1)
        df[f'std_{w}w_ago'] = df[cols].std(axis = 1)
        df[f'q1_{w}w_ago'] = df[cols].quantile(0.25, axis = 1)
        df[f'q2_{w}w_ago'] = df[cols].quantile(0.50, axis = 1)
        df[f'q3_{w}w_ago'] = df[cols].quantile(0.75, axis = 1)

    # Dtype Cleanup
    cat_col = ['quarter', 'month', 'day_of_week', 'week_of_year', 'season']
    df[cat_col] = df[cat_col].astype('category')

    # Time Index
    df['time_idx'] = (df['date'] - df['date'].min()).dt.days

    return df

In [5]:
# ==================================================
# Feature Extractor for TFT
# ==================================================

def TFTFeatures(df: pd.DataFrame):
    # Column Organization
    df = df.rename(columns = {'영업일자': 'date', '영업장명_메뉴명': 'store_menu', '매출수량': 'qty'})
    df[['store', 'menu']] = df['store_menu'].str.split('_', n = 1, expand = True, regex = False)
    df['date'] = pd.to_datetime(df['date'], format = '%Y-%m-%d')

    # Quarter: 1, 2, 3, 4
    df['quarter'] = df['date'].dt.quarter

    # Month: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12
    df['month'] = df['date'].dt.month

    # Day of the Week: 1, 2, 3, 4, 5, 6, 7
    df['day_of_week'] = df['date'].dt.day_of_week + 1

    # Week of the Year: 1, ..., 53
    iso = df['date'].dt.isocalendar()
    df['week_of_year'] = iso.week.astype(int)

    # Season: 1, 2, 3, 4
    df['season'] = np.select([
        df['month'].isin([3, 4, 5]),
        df['month'].isin([6, 7, 8]),
        df['month'].isin([9, 10, 11]),
        df['month'].isin([12, 1, 2])
    ], [1, 2, 3, 4])

    # Peak: 0-1
    df['is_peak'] = df['month'].isin([1, 2]).astype(int)

    # Weekend: 0-1
    df['is_weekend'] = df['day_of_week'].isin([6, 7]).astype(int)

    # Holiday: 0-1
    years = df['date'].dt.year.unique()
    df['is_holiday'] = (df['date'].dt.date.isin(holidays.KR(years = years))).astype(int)
    
    # Potential Group Menu: 0-1
    df['potential_group_menu'] = df['menu'].str.contains(
        '단체|Conference|room|OPUS|이용료|Cookie Platter', regex = True
    ).astype(int)

    # Holidays Left in the Year
    df_list = []
    for year in years:
        df_tmp = pd.DataFrame({'date': pd.date_range(start = f'{year}-01-01', end = f'{year}-12-31')})
        df_tmp['is_holiday'] = (df_tmp['date'].dt.date.isin(holidays.KR(years = [year])))
        df_tmp['cum_holiday'] = df_tmp['is_holiday'].cumsum()
        df_tmp['holidays_left_in_year'] = (
            df_tmp['cum_holiday'].max()
            - df_tmp['cum_holiday']
            + df_tmp['is_holiday']
        )
        df_list.append(df_tmp)
    df_tmp = pd.concat(df_list, axis = 0)
    df = df.merge(df_tmp[['date', 'holidays_left_in_year']], 'left', 'date')

    # Isolated Holiday: 0-1
    df_tmp = pd.DataFrame({'date': pd.date_range(
            start = df['date'].min() - pd.Timedelta(days = 30),
            end = df['date'].max() + pd.Timedelta(days = 30)
    )})
    df_tmp['is_holiday'] = df_tmp['date'].dt.date.isin(holidays.KR(years = df_tmp['date'].dt.year.unique()))
    df_tmp['prev_holiday'] = df_tmp['is_holiday'].shift(1, fill_value = 0)
    df_tmp['next_holiday'] = df_tmp['is_holiday'].shift(-1, fill_value = 0)
    df_tmp['isolated_holiday'] = (
        (df_tmp['date'].dt.day_of_week.isin([1, 2, 3])) &
        (df_tmp['is_holiday'] == 1) &
        (df_tmp['prev_holiday'] == 0) &
        (df_tmp['next_holiday'] == 0)).astype(int)
    df = df.merge(df_tmp[['date', 'isolated_holiday']], 'left', 'date')

    # Holidays Left in Streak
    series_tmp = df_tmp.groupby('date', as_index = True)['is_holiday'].max().sort_index()
    series_tmp_rev = series_tmp.iloc[::-1]
    series_tmp = (
        series_tmp_rev.cumsum() - series_tmp_rev.cumsum().where(~series_tmp_rev).ffill().fillna(0)
    ).iloc[::-1]
    df_tmp = series_tmp.rename('holidays_left_in_streak').reset_index().rename(columns = {'index': 'date'})
    df = df.merge(df_tmp, 'left', 'date')
    df['holidays_left_in_streak'] = df['holidays_left_in_streak'].astype(int)

    # Keep Default Sorting
    df = df.sort_values(['store_menu', 'date']).reset_index(drop = True)

    # Dtype Cleanup
    str_col = ['store', 'menu', 'quarter', 'month', 'day_of_week', 'week_of_year', 'season']
    df[str_col] = df[str_col].astype(str)
    num_col = [
        'qty', 'is_peak', 'is_weekend', 'is_holiday', 'potential_group_menu',
        'holidays_left_in_year', 'isolated_holiday', 'holidays_left_in_streak'
    ]
    df[num_col] = df[num_col].astype(float)

    # Time Index
    df['time_idx'] = (df['date'] - df['date'].min()).dt.days

    return df

In [6]:
# ==================================================
# Test DataFrame Date Extender
# ==================================================

def TestExtender(df: pd.DataFrame, test_size):
    df_tmp = df.copy()
    df_tmp['영업일자'] = pd.to_datetime(df_tmp['영업일자'], format = '%Y-%m-%d')
    max_date = df_tmp['영업일자'].max()

    join_date = pd.DataFrame({
        '영업일자': pd.date_range(
            start = max_date + pd.Timedelta(days = 1),
            end = max_date + pd.Timedelta(days = test_size)
        )
    })
    join_category = pd.DataFrame({'영업장명_메뉴명': df_tmp['영업장명_메뉴명'].unique()})
    join_future = join_date.merge(join_category, 'cross')
    df_tmp = pd.concat([df_tmp, join_future], axis = 0).sort_values(
        ['영업일자', '영업장명_메뉴명'], ignore_index = True
    )

    return df_tmp

In [7]:
# ==================================================
# Cross-Validator for Tree-Based Models
# ==================================================

class TreeCV(BaseCrossValidator):
    def __init__(
        self,
        n_splits: int, # Number of Splits
        validation_size: int, # Days to Validate
        time_idx: np.ndarray, # Time Index Column Required
    ):
        self.n_splits = int(n_splits)
        self.validation_size = int(validation_size)
        self.time_idx = np.asarray(time_idx)

        # Message on Failure
        if self.time_idx.ndim != 1:
            raise ValueError('time_idx must be 1-D.')
        if self.n_splits < 1 or self.validation_size < 1:
            raise ValueError('n_splits and validation_size must be >= 1.')
        if np.unique(self.time_idx).size < self.validation_size * self.n_splits + 1:
            raise ValueError('Not enough distinct time points for requested splits.')
        
        # Compute Splits in Advance for Computational Efficiency
        max_t = self.time_idx.max()
        self._splits: list[tuple[np.ndarray, np.ndarray]] = []
        
        for k in range(1, self.n_splits + 1):
            validation_end = max_t - self.validation_size * (k - 1)
            validation_start = max_t - self.validation_size * k + 1

            validation_idx = (self.time_idx >= validation_start) & (self.time_idx <= validation_end)
            train_idx = (self.time_idx < validation_start)

            validation_idx = np.flatnonzero(validation_idx)
            train_idx = np.flatnonzero(train_idx)

            self._splits.append((train_idx, validation_idx))
            
    def get_n_splits(self, X = None, y = None, groups = None):
        return len(self._splits)

    def split(self, X: pd.DataFrame, y = None, groups = None) -> Iterable[tuple[np.ndarray, np.ndarray]]:
        for train_idx, validation_idx in self._splits:
            yield train_idx, validation_idx

In [8]:
# ==================================================
# Cross-Validator for TFT
# ==================================================

class TFTCV(BaseCrossValidator):
    def __init__(
        self,
        n_splits: int, # Number of Splits
        input_size: int, # Number of Input Days
        validation_size: int, # Days to Validate
        time_idx: np.ndarray, # Time Index Column Required
    ):
        self.n_splits = int(n_splits)
        self.input_size = int(input_size)
        self.validation_size = int(validation_size)
        self.time_idx = np.asarray(time_idx)

        # Message on Failure
        if self.time_idx.ndim != 1:
            raise ValueError('time_idx must be 1-D.')
        if self.n_splits < 1 or self.validation_size < 1:
            raise ValueError('n_splits and validation_size must be >= 1.')
        if np.unique(self.time_idx).size < self.validation_size * self.n_splits + self.input_size + 1:
            raise ValueError('Not enough distinct time points for requested splits.')
        
        # Compute Splits in Advance for Computational Efficiency
        max_t = self.time_idx.max()
        self._splits: list[tuple[np.ndarray, np.ndarray]] = []
        
        for k in range(1, self.n_splits + 1):
            validation_end = max_t - self.validation_size * (k - 1)
            validation_start = max_t - self.validation_size * k - self.input_size + 1
            train_end = max_t - self.validation_size * k + 1

            validation_idx = (self.time_idx >= validation_start) & (self.time_idx <= validation_end)
            train_idx = (self.time_idx <= train_end)

            validation_idx = np.flatnonzero(validation_idx)
            train_idx = np.flatnonzero(train_idx)

            self._splits.append((train_idx, validation_idx))
            
    def get_n_splits(self, X = None, y = None, groups = None):
        return len(self._splits)

    def split(self, X: pd.DataFrame, y = None, groups = None) -> Iterable[tuple[np.ndarray, np.ndarray]]:
        for train_idx, validation_idx in self._splits:
            yield train_idx, validation_idx

In [9]:
# ==================================================
# Scikit-Learn-Compatible SMAPE
# ==================================================

# SMAPE Error: [0, 2]
def smape_(y_true, y_pred, eps = 1e-8):
    y_true = np.array(y_true, dtype = float)
    y_pred = np.array(y_pred, dtype = float)

    return np.mean(2.0 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + eps))

# SMAPE Score: [-2, 0]
smape = make_scorer(smape_, greater_is_better = False)

# Tree-Based Models

In [ ]:
# ==================================================
# Train Data Preprocessing
# ==================================================

# Load Train Data
train = pd.read_csv('train/train.csv')

# Correct Negative Quantities
train['매출수량'] = train['매출수량'].abs()

# Extract Features
train = TreeFeatures(train)

# Prepare Data for Clustering
train_tmp = train.copy().drop(
    ['date', 'store_menu', 'qty', 'time_idx'], axis = 1
).dropna()
train_tmp = pd.get_dummies(train_tmp)

dummy_col = train_tmp.columns
join_idx = train_tmp.index

scaler = StandardScaler().set_output(transform = 'pandas')
train_tmp = scaler.fit_transform(train_tmp)

# Apply K-Means Clustering
cluster = KMeans(n_clusters = CFG['N_CLUSTERS'], random_state = CFG['SEED'])
join_cluster = pd.Series(
    cluster.fit_predict(train_tmp),
    index = join_idx,
    name = 'cluster'
).astype('category')

# Attach Cluster Labels
train = train.join(join_cluster)

# Save Preprocessed Train Data
joblib.dump(train, 'train/TRAIN_TREE.joblib')

# Save Clustering Artifacts
joblib.dump(dummy_col, 'artifacts/cluster_dummy_col.joblib')
joblib.dump(scaler, 'artifacts/cluster_scaler.joblib')
joblib.dump(cluster, 'artifacts/cluster_kmeans.joblib')

In [ ]:
# ==================================================
# Test Data Preprocessing
# ==================================================

# Load Clustering Artifacts
dummy_col = joblib.load('artifacts/cluster_dummy_col.joblib')
scaler = joblib.load('artifacts/cluster_scaler.joblib')
cluster = joblib.load('artifacts/cluster_kmeans.joblib')

# Load Test Data
test = []
for i in range(10):
    test_tmp = pd.read_csv(f'test/TEST_0{i}.csv')

    # Extend Dates
    test_tmp = TestExtender(test_tmp, CFG['PREDICT'])

    # Extract Features
    test_tmp = TreeFeatures(test_tmp)

    # Keep Future Rows Only
    test_tmp = test_tmp.drop('qty', axis = 1).dropna().reset_index(drop = True)

    # Prepare Data for Clustering
    test_tmp2 = test_tmp.copy().drop(['date', 'store_menu', 'time_idx'], axis = 1)
    test_tmp2 = pd.get_dummies(test_tmp2).reindex(columns = dummy_col, fill_value = 0)
    test_tmp2 = scaler.transform(test_tmp2)

    # Apply K-Means Clustering
    test_tmp['cluster'] = cluster.predict(test_tmp2)
    test_tmp['cluster'] = test_tmp['cluster'].astype('category')

    # Accumulate Test DataFrames
    test.append(test_tmp)

# Save Preprocessed Test Data
joblib.dump(test, 'test/TEST_TREE.joblib')

In [ ]:
# ==================================================
# Base Estimator Definitions
# ==================================================

# Load Preprocessed Train Data
train = joblib.load('train/TRAIN_TREE.joblib')

# Split Features and Target
x = train.drop('qty', axis = 1)
y = train['qty']

# Define Column Groups
drop_col = ['date', 'store_menu', 'time_idx']
num_col = x.select_dtypes(include = 'number').columns.drop('time_idx').to_list()
cat_col = x.select_dtypes(include = 'category').columns.to_list()

# Define Preprocessor
prep = ColumnTransformer([
    ('drop', 'drop', drop_col),
    ('num', 'passthrough', num_col),
    ('cat', TargetEncoder(target_type = 'continuous', shuffle = False), cat_col)
]).set_output(transform = 'pandas')

# Define Base Estimators
models = {}

models['lgbm'] = Pipeline([
    ('prep', prep),
    ('model', LGBMRegressor(
        boosting_type = 'gbdt',
        objective = 'tweedie',
        random_state = CFG['SEED'],
        device = CFG['DEVICE'],
        verbose = -1
    ))
])

models['xgb'] = Pipeline([
    ('prep', prep),
    ('model', XGBRegressor(
        booster = 'gbtree',
        objective = 'reg:tweedie',
        random_state = CFG['SEED'],
        device = CFG['DEVICE'],
        verbosity = 0
    ))
])

models['rf'] = Pipeline([
    ('prep', prep),
    ('model', RandomForestRegressor(
        criterion = 'poisson',
        random_state = CFG['SEED'],
        bootstrap = False,
        n_jobs = -1
    ))
])

# Cross-Validator
cv = TreeCV(
    n_splits = CFG['K_VALIDATIONS'],
    validation_size = CFG['PREDICT'],
    time_idx = x['time_idx']
)

In [ ]:
# ==================================================
# Hyperparameter Tuning
# ==================================================

# Define Per-Model Optuna Objective
for i, j in models.items():
    def objective(trial):
        if i == 'lgbm':
            params = {
                'model__tweedie_variance_power': trial.suggest_float('model__tweedie_variance_power', 1.5, 2.0),
                'model__num_leaves': trial.suggest_int('model__num_leaves', 32, 256),
                'model__max_depth': trial.suggest_int('model__max_depth', 4, 16),
                'model__learning_rate': trial.suggest_float('model__learning_rate', 1e-3, 0.2, log = True),
                'model__n_estimators': trial.suggest_int('model__n_estimators', 300, 2000),
                'model__min_split_gain': trial.suggest_float('model__min_split_gain', 0.0, 0.1),
                'model__min_child_samples': trial.suggest_int('model__min_child_samples', 50, 300),
                'model__colsample_bytree': trial.suggest_float('model__colsample_bytree', 0.5, 1.0),
                'model__reg_alpha': trial.suggest_float('model__reg_alpha', 0.1, 10.0, log = True),
                'model__reg_lambda': trial.suggest_float('model__reg_lambda', 0.1, 10.0, log = True)
            }
            
        if i == 'xgb':
            params = {
                'model__tweedie_variance_power': trial.suggest_float('model__tweedie_variance_power', 1.5, 2.0),
                'model__max_leaves': trial.suggest_int('model__max_leaves', 32, 256),
                'model__max_depth': trial.suggest_int('model__max_depth', 4, 16),
                'model__learning_rate': trial.suggest_float('model__learning_rate', 1e-3, 0.2, log = True),
                'model__n_estimators': trial.suggest_int('model__n_estimators', 300, 2000),
                'model__gamma': trial.suggest_float('model__gamma', 0, 0.1),
                'model__min_child_weight': trial.suggest_int('model__min_child_weight', 1, 20),
                'model__colsample_bytree': trial.suggest_float('model__colsample_bytree', 0.5, 1.0),
                'model__reg_alpha': trial.suggest_float('model__reg_alpha', 0.1, 10.0, log = True),
                'model__reg_lambda': trial.suggest_float('model__reg_lambda', 0.1, 10.0, log = True)
            }

        if i == 'rf':
            params = {
                'model__n_estimators': trial.suggest_int('model__n_estimators', 300, 2000),
                'model__max_features': trial.suggest_float('model__max_features', 0.2, 0.8),
                'model__min_impurity_decrease': trial.suggest_float('model__min_impurity_decrease', 0, 0.1)
            }

        model = clone(j).set_params(**params)

        errors = cross_val_score(
            model, x, y, scoring = smape, cv = cv, n_jobs = 1
        )

        return -errors.mean()
    
    # Lock Seed
    SeedLocker()

    # Run Optuna
    study = optuna.create_study(
        direction = 'minimize',
        sampler = optuna.samplers.TPESampler(seed = CFG['SEED'])
    )
    study.optimize(objective, n_trials = 30, n_jobs = 1, show_progress_bar = True)

    # Report Best Trial
    print(f'\nBest Parameters (\'{i}\')):')
    print(study.best_params)
    print(f'Best CV SMAPE (\'{i}\'):')
    print(study.best_trial.value)
    print('\n========================================\n')

    # Save Best Parameters and CV Error
    with open(f'params/params_{i}.json', 'w') as f:
        json.dump(study.best_trial.params, f, indent = 4)
    with open(f'cv_errors/cv_error_{i}.txt', 'w') as f:
        f.write(str(study.best_trial.value))

In [ ]:
# ==================================================
# OOF Predictions
# ==================================================

# Load and Apply Best Parameters
for i, j in models.items():
    with open(f'params/params_{i}.json', 'r') as f:
        params_tmp = json.load(f)
    j.set_params(**params_tmp)

# Generate OOF Meta-Features
oof_tree = []
for k, (t, v) in enumerate(cv.split(x, y)):
    t_x, v_x = x.iloc[t], x.iloc[v]
    t_y, v_y = y.iloc[t], y.iloc[v]

    # Clone and Train Base Models
    SeedLocker()
    models_tmp = {i: clone(j).fit(t_x, t_y) for i, j in models.items()}

    # Compute Meta-Features
    meta_tmp = np.column_stack([models_tmp[i].predict(v_x) for i in models_tmp.keys()])

    # Accumulate (Meta, Target) Tuples
    oof_tree.append((meta_tmp, np.array(v_y)))

    # Show Progress
    print(f'Fold {k + 1} done.')

# Save OOF Meta-Features
joblib.dump(oof_tree, 'artifacts/OOF_TREE.joblib')

# TFT

In [ ]:
# ==================================================
# Train Data Preprocessing
# ==================================================

# Lock Seed Before Building DataLoaders
SeedLocker()

# Load Train Data
train = pd.read_csv('train/train.csv')

# Correct Negative Quantities
train['매출수량'] = train['매출수량'].abs()

# Extract Features
train = TFTFeatures(train)

# Define Column Groups
time_idx = 'time_idx'
target = 'qty'
group_ids = ['store', 'menu']
static_categoricals = ['store', 'menu']
static_reals = ['potential_group_menu']
time_varying_known_categoricals = ['quarter', 'month', 'day_of_week', 'week_of_year', 'season']
time_varying_known_reals = [
    'is_peak', 'is_weekend', 'is_holiday', 'holidays_left_in_year',
    'isolated_holiday', 'holidays_left_in_streak'
]
time_varying_unknown_reals = ['qty']

# Create a Train-Validation Split
train_end = train['time_idx'].max() - CFG['K_VALIDATIONS'] * CFG['PREDICT']
valid_start = train['time_idx'].max() - CFG['K_VALIDATIONS'] * CFG['PREDICT'] - CFG['LOOKBACK'] + 1
train_df = train.loc[train['time_idx'] <= train_end]
valid_df = train.loc[train['time_idx'] >= valid_start]

# Build Training Dataset
train_set = TimeSeriesDataSet(
    data = train_df,
    time_idx = time_idx,
    target = target,
    group_ids = group_ids,
    max_encoder_length = CFG['LOOKBACK'],
    max_prediction_length = CFG['PREDICT'],
    static_categoricals = static_categoricals,
    static_reals = static_reals,
    time_varying_known_categoricals = time_varying_known_categoricals,
    time_varying_known_reals = time_varying_known_reals,
    time_varying_unknown_reals = time_varying_unknown_reals,
    target_normalizer = GroupNormalizer(
        groups = group_ids,
        transformation = 'relu'
    ),
    add_relative_time_idx = True,
    add_target_scales = True,
    add_encoder_length = True
)

# Save Training Dataset for Later Use
train_set.save('train/TRAIN_DATASET.tsd')

# Derive Validation Dataset From Training Dataset
valid_set = TimeSeriesDataSet.from_dataset(
    dataset = train_set,
    data = valid_df,
    stop_randomization = True,
    predict = False,    # Setting True Creates a Single Sequence
    min_prediction_idx = valid_df['time_idx'].min() + CFG['LOOKBACK']
)

# Build Training DataLoader
train_loader = train_set.to_dataloader(
    train = True,
    batch_size = CFG['BATCH'],
    # num_workers = 6,
    # prefetch_factor = 4,
    pin_memory = True
)

# Build Validation DataLoader
valid_loader = valid_set.to_dataloader(
    train = False,
    batch_size = CFG['BATCH'],
    # num_workers = 6,
    # prefetch_factor = 4,
    pin_memory = True
)

In [ ]:
# ==================================================
# Test Data Preprocessing
# ==================================================

# Load Training Dataset
train_set = torch.load('train/TRAIN_DATASET.tsd', weights_only = False)

# Load Test Data
for i in range(10):
    test_tmp = pd.read_csv(f'test/TEST_0{i}.csv')

    # Extend Dates
    test_tmp = TestExtender(test_tmp, CFG['PREDICT'])

    # Extract Features
    test_tmp = TFTFeatures(test_tmp).fillna(0.0)

    # Derive Test Dataset From Training Dataset
    test_set_tmp = TimeSeriesDataSet.from_dataset(
        dataset = train_set,
        data = test_tmp,
        stop_randomization = True,
        predict = True,
        min_prediction_idx = test_tmp['time_idx'].max() - CFG['PREDICT'] + 1
    )

    # Save Test Datasets
    test_set_tmp.save(f'test/TEST_DATASET_0{i}.tsd')

In [ ]:
# ==================================================
# Hyperparameter Tuning and Training
# ==================================================

# Lock Seed
SeedLocker()

# Define Optuna Objective
def objective(trial):
    dropout = trial.suggest_float('dropout', 0.1, 0.3)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log = True)
    weight_decay = trial.suggest_float('weight_decay', 1e-4, 1e-2, log = True)
    gradient_clip_val = trial.suggest_float('gradient_clip_val', 0.5, 1.0)

    # Define TFT
    model = TemporalFusionTransformer.from_dataset(
        dataset = train_set,
        hidden_size = 128,
        lstm_layers = 2,
        dropout = dropout,
        output_size = 3,
        loss = QuantileLoss([0.1, 0.5, 0.9]),
        attention_head_size = 4,
        hidden_continuous_size = 64,
        learning_rate = learning_rate,
        weight_decay = weight_decay,
        mask_bias = float('-inf'),
        logging_metrics = [SMAPE()],
        optimizer = 'adamw',
        reduce_on_plateau_patience = 3
    )

    # Set Up Early Stopping Callback
    early = EarlyStopping(
        monitor = 'val_SMAPE',
        patience = 7,
        verbose = True,
        mode = 'min'
    )

    # Set Up Model Checkpointing
    ckpt = ModelCheckpoint(
        monitor = 'val_SMAPE',
        mode = 'min',
        save_last = True,
        save_top_k = 1,
        dirpath = 'artifacts/tft/',
        filename = 'tft-{epoch:02d}-{val_SMAPE:.4f}'
    )

    # Initialize Trainer
    trainer = Trainer(
        accelerator = CFG['DEVICE'],
        devices = 1,
        precision = '16-mixed',
        logger = False,
        callbacks = [early, ckpt],
        max_epochs = CFG['EPOCH'],
        enable_progress_bar = True,
        gradient_clip_val = gradient_clip_val
    )

    # Fit TFT
    trainer.fit(
        model = model,
        train_dataloaders = train_loader,
        val_dataloaders = valid_loader
    )

    # Store Best Checkpoint Path as an Attribute
    best_path = os.path.relpath(ckpt.best_model_path, start = os.getcwd())
    trial.set_user_attr('best_tft_path', best_path)

    return float(ckpt.best_model_score.detach().cpu().item())

# Run Optuna
study = optuna.create_study(
    direction = 'minimize',
    sampler = optuna.samplers.TPESampler(seed = CFG['SEED'])
)
study.optimize(objective, n_trials = 15, n_jobs = 1, show_progress_bar = True)

# Report Best Trial
print('\nBest Parameters:')
print(study.best_params)
print('Best CV SMAPE:')
print(study.best_trial.value)
# print('* SMAPE in PyTorch Forecasting includes true zeros, leading to overestimation vs. tree-based models.')
print('\n========================================\n')

# Save Best CV Error
with open('cv_errors/cv_error_tft.txt', 'w') as f:
    f.write(str(study.best_trial.value))

# Save Path to Best Checkpoint
with open('artifacts/best_tft_path.txt', 'w') as f:
    f.write(study.best_trial.user_attrs['best_tft_path'])

In [ ]:
# ==================================================
# OOF Prediction
# ==================================================

# Load Training Dataset
train_set = torch.load('train/TRAIN_DATASET.tsd', weights_only = False)

# Load Best TFT Checkpoint
with open('artifacts/best_tft_path.txt', 'r') as f:
    best_tft_path = f.readline()
tft = TemporalFusionTransformer.load_from_checkpoint(best_tft_path)
tft.output_transformer = getattr(train_set, 'target_normalizer', None)

# Match Validation Folds to Tree-Based Models
cv = TFTCV(
    n_splits = CFG['K_VALIDATIONS'],
    input_size = CFG['LOOKBACK'],
    validation_size = CFG['PREDICT'],
    time_idx = train['time_idx']
)

# Generate OOF Meta-Features
oof_tft = []
for _, v in cv.split(train):
    oof_df = train.iloc[v]

    # Derive OOF Dataset From Training Dataset
    oof_set_tmp = TimeSeriesDataSet.from_dataset(
        dataset = train_set,
        data = oof_df,
        stop_randomization = True,
        predict = True,
        min_prediction_idx = oof_df['time_idx'].max() - CFG['PREDICT'] + 1
    )

    # Build OOF DataLoader
    oof_loader_tmp = oof_set_tmp.to_dataloader(
        train = False,
        batch_size = CFG['BATCH'],
        # num_workers = 6,
        # prefetch_factor = 4,
        pin_memory = True
    )

    # Compute Meta-Features
    meta_tmp = tft.predict(oof_loader_tmp).reshape(-1).cpu().numpy()

    # Accumulate Meta-Feature Arrays
    oof_tft.append(meta_tmp)

# Save OOF Meta-Feature
joblib.dump(oof_tft, 'artifacts/OOF_TFT.joblib')

# Ensemble

In [10]:
# ==================================================
# Stacking with Ridge Regression
# ==================================================

# Define Unweighted Version of Competition Metric
# SMAPE Error Excluding Zero Targets: [0, 2]
def smape2_(y_true, y_pred):
    y_true = np.array(y_true, dtype = float)
    y_pred = np.array(y_pred, dtype = float)

    mask = (y_true != 0)
    if not any(mask):
        return 0.0
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    return np.mean(2.0 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred)))

# SMAPE Score Excluding Zero Targets: [-2, 0]
smape2 = make_scorer(smape2_, greater_is_better = False)

# Load OOF Meta-Features
oof_tree = joblib.load('artifacts/OOF_TREE.joblib')
oof_tft = joblib.load('artifacts/OOF_TFT.joblib')

# Combine Predictions of Individual Models
all_preds = []
all_trues = []
for (y_tree, y_true), y_tft in zip(oof_tree, oof_tft):
    preds_tmp = np.column_stack([y_tree, y_tft])
    all_preds.append(preds_tmp)
    all_trues.append(y_true)
all_preds = np.concatenate(all_preds, axis = 0)
all_trues = np.concatenate(all_trues, axis = 0)

# Define Meta Model
ridge = RidgeCV(
    alphas = np.logspace(0, 7, 1000),
    scoring = smape2
)

# Perform Ridge LOOCV
ridge.fit(all_preds, all_trues)

# Report Best Alpha and CV Error
print('Best Alpha')
print(ridge.alpha_)
print('Best CV SMAPE:')
print(-ridge.best_score_)

# Save Meta Model
joblib.dump(ridge, 'artifacts/ensemble_ridge.joblib')

# Save SMAPE Excluding Zero Targets
with open('cv_errors/cv_error_ensemble.txt', 'w') as f:
    f.write(str(-ridge.best_score_))

Best Alpha
1079028.7915161836
Best CV SMAPE:
0.5262616322519291


# Inference

In [11]:
# ==================================================
# Inference: Tree-Based Models 
# ==================================================

# Load Preprocessed Train/Test Data
train = joblib.load('train/TRAIN_TREE.joblib')
test = joblib.load('test/TEST_TREE.joblib')

# Split Features and Target
x = train.drop('qty', axis = 1)
y = train['qty']

# Define Column Groups
drop_col = ['date', 'store_menu', 'time_idx']
num_col = x.select_dtypes(include = 'number').columns.drop('time_idx').to_list()
cat_col = x.select_dtypes(include = 'category').columns.to_list()

# Define Preprocessor
prep = ColumnTransformer([
    ('drop', 'drop', drop_col),
    ('num', 'passthrough', num_col),
    ('cat', TargetEncoder(target_type = 'continuous', shuffle = False), cat_col)
]).set_output(transform = 'pandas')

# Define Base Estimators
models = {}

models['lgbm'] = Pipeline([
    ('prep', prep),
    ('model', LGBMRegressor(
        boosting_type = 'gbdt',
        objective = 'tweedie',
        random_state = CFG['SEED'],
        device = CFG['DEVICE'],
        verbose = -1
    ))
])

models['xgb'] = Pipeline([
    ('prep', prep),
    ('model', XGBRegressor(
        booster = 'gbtree',
        objective = 'reg:tweedie',
        random_state = CFG['SEED'],
        device = CFG['DEVICE'],
        verbosity = 0
    ))
])

models['rf'] = Pipeline([
    ('prep', prep),
    ('model', RandomForestRegressor(
        criterion = 'poisson',
        random_state = CFG['SEED'],
        bootstrap = False,
        n_jobs = -1
    ))
])

# Refit Tree-Based Models Using Best Parameters
for i, j in models.items():
    SeedLocker()
    with open(f'params/params_{i}.json', 'r') as f:
        params_tmp = json.load(f)
    j.set_params(**params_tmp)
    j.fit(x, y)

# Run Inference With Tree-Based Models
preds_tree = []
for i in test:
    pred_tmp = np.column_stack([models[j].predict(i) for j in models.keys()])

    # Keep the Original Format
    df_tmp = i.loc[:, ['date', 'store_menu']]
    df_tmp[[j for j in models.keys()]] = pred_tmp

    # Accumulate Predictions
    preds_tree.append(df_tmp)

# Concatenate Predictions
pred = pd.concat(preds_tree, axis = 0, ignore_index = True)

Seed set to 72
Seed set to 72
Seed set to 72


In [12]:
# ==================================================
# Inference: TFT
# ==================================================

# Load Training Dataset
train_set = torch.load('train/TRAIN_DATASET.tsd', weights_only = False)

# Load Best TFT Checkpoint
with open('artifacts/best_tft_path.txt', 'r') as f:
    best_tft_path = f.readline()
tft = TemporalFusionTransformer.load_from_checkpoint(best_tft_path)
tft.output_transformer = getattr(train_set, 'target_normalizer', None)

# Run Inference With TFT
pred_tft = []
for i in range(10):
    test_set_tmp = torch.load(f'test/TEST_DATASET_0{i}.tsd', weights_only = False)

    # Build Test DataLoader
    test_loader_tmp = test_set_tmp.to_dataloader(
        train = False,
        batch_size = CFG['BATCH'],
        # num_workers = 6,
        # prefetch_factor = 4,
        pin_memory = True
    )

    # Compute Meta-Features
    pred_tmp = tft.predict(test_loader_tmp).reshape(-1).cpu().numpy()

    # Accumulate Predictions
    pred_tft.append(pred_tmp)

# Combine TFT and Tree-Based Predictions
pred_tft = np.concatenate(pred_tft, axis = 0)
pred['tft'] = pred_tft

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
💡 Tip: For

In [13]:
# ==================================================
# Inference: Ensemble
# ==================================================

# Load Meta Model
ridge = joblib.load('artifacts/ensemble_ridge.joblib')

# Predict with Meta Model
pred['ensemble'] = np.round(ridge.predict(pred[['lgbm', 'xgb', 'rf', 'tft']]))
pred = pred.drop(['lgbm', 'xgb', 'rf', 'tft'], axis = 1)

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but RidgeCV was fitted without feature names
  warnings.warn(


In [14]:
# ==================================================
# Submission
# ==================================================

# Pivot Predictions to Wide Submission Format
submission = pred.pivot(
    index = 'date', columns = 'store_menu', values = 'ensemble'
).reset_index(names = ['영업일자']).sort_values('영업일자').rename_axis(columns = None)

# Match Sample Submission Schema
sample_submission = pd.read_csv('sample_submission.csv')
sample_col = sample_submission.columns
sample_idx = sample_submission['영업일자']

submission = submission[sample_col]
submission['영업일자'] = sample_idx

# Save Submission CSV
submission.to_csv('submission.csv', index = False)